In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

DATA_DIR = Path("/Volumes/T7/DS Projects/Chicago Ride Demand Forecasting/data/processed")

df_h3 = pd.read_parquet(DATA_DIR / "chirde.h3_hourly_comp.parquet").copy()
df_city = pd.read_parquet(DATA_DIR / "chirde.citywide_hourly.parquet").copy()

# ---- detect important columns ----
h3_col = None
for c in df_h3.columns:
    if c == "h3_index":
        h3_col = c
        break
if h3_col is None:
    for c in df_h3.columns:
        if "h3" in c.lower():
            h3_col = c
            break

dt_h3 = None
for c in df_h3.columns:
    if pd.api.types.is_datetime64_any_dtype(df_h3[c]):
        dt_h3 = c
        break
if dt_h3 is None:
    for c in df_h3.columns:
        if "date" in c.lower() or "time" in c.lower():
            dt_h3 = c
            break

dt_city = None
for c in df_city.columns:
    if pd.api.types.is_datetime64_any_dtype(df_city[c]):
        dt_city = c
        break
if dt_city is None:
    for c in df_city.columns:
        if "date" in c.lower() or "time" in c.lower():
            dt_city = c
            break

print("Detected:")
print(" h3_col  =", h3_col)
print(" dt_h3   =", dt_h3)
print(" dt_city =", dt_city)

if h3_col is None or dt_h3 is None or dt_city is None:
    raise ValueError("Could not auto-detect key columns")

df_h3[dt_h3] = pd.to_datetime(df_h3[dt_h3])
df_city[dt_city] = pd.to_datetime(df_city[dt_city])

# ---- add hour key ----
df_h3["hour_key"] = df_h3[dt_h3].dt.hour
df_city["hour_key"] = df_city[dt_city].dt.hour

# ---- numeric columns only ----
num_h3 = df_h3.select_dtypes(include=[np.number]).columns.tolist()
num_city = df_city.select_dtypes(include=[np.number]).columns.tolist()

print("\nNumeric H3 cols:", len(num_h3))
print("Numeric City cols:", len(num_city))

# ---- build caches ----
h3_hour_df = df_h3.groupby([h3_col, "hour_key"])[num_h3].median()
h3_df = df_h3.groupby(h3_col)[num_h3].median()
city_hour_df = df_city.groupby("hour_key")[num_city].median()

h3_hour_dict = {
    f"{hex_id}_{hour}": row.to_dict()
    for (hex_id, hour), row in h3_hour_df.iterrows()
}
h3_dict = {
    hex_id: row.to_dict()
    for hex_id, row in h3_df.iterrows()
}
city_hour_dict = {
    hour: row.to_dict()
    for hour, row in city_hour_df.iterrows()
}

feature_cache = {
    "h3_hour": h3_hour_dict,
    "h3": h3_dict,
    "h3_global": df_h3[num_h3].median().to_dict(),
    "city_hour": city_hour_dict,
    "city_global": df_city[num_city].median().to_dict(),
    "meta": {
        "h3_col": h3_col,
        "dt_h3": dt_h3,
        "dt_city": dt_city,
        "n_h3_hour": len(h3_hour_dict),
        "n_h3": len(h3_dict),
        "n_city_hour": len(city_hour_dict),
    }
}

cache_path = DATA_DIR / "feature_cache.joblib"
joblib.dump(feature_cache, cache_path, compress=3)

print("\n✅ Saved:", cache_path)
print("h3_hour entries:", len(h3_hour_dict))
print("h3 entries:", len(h3_dict))
print("city_hour entries:", len(city_hour_dict))

Detected:
 h3_col  = h3_index
 dt_h3   = hour_bucket
 dt_city = hour_bucket

Numeric H3 cols: 76
Numeric City cols: 79

✅ Saved: /Volumes/T7/DS Projects/Chicago Ride Demand Forecasting/data/processed/feature_cache.joblib
h3_hour entries: 13008
h3 entries: 542
city_hour entries: 24


In [3]:
import os, time, json, math, joblib, threading, subprocess
import numpy as np
from pathlib import Path
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import List, Optional
from datetime import datetime as dt_type
import uvicorn
import requests

PROJECT_ROOT = "/Volumes/T7/DS Projects/Chicago Ride Demand Forecasting"
os.chdir(PROJECT_ROOT)

# kill old server
subprocess.run(["pkill", "-9", "-f", "uvicorn"], capture_output=True)
time.sleep(2)

BEST_DIR = Path("models/best")
DATA_DIR = Path("data/processed")

# ---------- load models ----------
MODELS = {}
METADATA = {}
FEATURES = {}

for target in ["T1", "T2", "T3", "T4"]:
    target_dir = BEST_DIR / target
    if not target_dir.exists():
        continue
    for algo_dir in target_dir.iterdir():
        if not algo_dir.is_dir():
            continue
        mf = algo_dir / "model.joblib"
        mt = algo_dir / "metadata.json"
        if mf.exists() and mt.exists():
            MODELS[target] = joblib.load(mf)
            with open(mt) as f:
                METADATA[target] = json.load(f)
            FEATURES[target] = METADATA[target].get("feature_names", [])
            print(f"✅ {target}: {len(FEATURES[target])} features")
            break

# ---------- load REAL feature cache ----------
FCACHE = joblib.load(DATA_DIR / "feature_cache.joblib")
print("✅ Feature cache loaded")
print("   h3_hour:", len(FCACHE["h3_hour"]))
print("   h3:", len(FCACHE["h3"]))
print("   city_hour:", len(FCACHE["city_hour"]))

# warmup
for target in MODELS:
    X = np.zeros((3, len(FEATURES[target])))
    MODELS[target].predict(X)

print("✅ Models warmed up")


def build_features(dt, h3_index=None, temp=None, hum=None, wind=None, precip=None):
    """
    Build features using REAL cached values.
    """
    h = dt.hour
    dow = dt.weekday()
    dom = dt.day
    p = precip or 0.0

    f = {}

    # 1) start with city hour baseline
    if h in FCACHE["city_hour"]:
        f.update(FCACHE["city_hour"][h])
    else:
        f.update(FCACHE["city_global"])

    # 2) override with h3-specific values if available
    if h3_index is not None:
        h3_hour_key = f"{h3_index}_{h}"
        if h3_hour_key in FCACHE["h3_hour"]:
            f.update(FCACHE["h3_hour"][h3_hour_key])
        elif h3_index in FCACHE["h3"]:
            f.update(FCACHE["h3"][h3_index])
        else:
            f.update(FCACHE["h3_global"])

    # 3) force time-based features from the request datetime
    f["hour"] = float(h)
    f["day_of_week"] = float(dow)
    f["week_of_year"] = float(dt.isocalendar()[1])
    f["hour_sin"] = math.sin(2 * math.pi * h / 24)
    f["hour_cos"] = math.cos(2 * math.pi * h / 24)
    f["dow_sin"] = math.sin(2 * math.pi * dow / 7)
    f["dow_cos"] = math.cos(2 * math.pi * dow / 7)
    f["dom_sin"] = math.sin(2 * math.pi * dom / 31)
    f["dom_cos"] = math.cos(2 * math.pi * dom / 31)

    # 4) override weather from request
    f["temperature_f"] = temp if temp is not None else f.get("temperature_f", 50.0)
    f["humidity"] = hum if hum is not None else f.get("humidity", 72.0)
    f["wind_speed_kmh"] = (wind * 1.60934) if wind is not None else f.get("wind_speed_kmh", 16.0)
    f["precipitation_mm"] = (p * 25.4)
    f["is_raining"] = 1.0 if p > 0 else 0.0

    # weather fallback fields
    if "snowfall_cm" not in f:
        f["snowfall_cm"] = 0.0
    if "is_snowing" not in f:
        f["is_snowing"] = 0.0
    if "weather_code" not in f:
        f["weather_code"] = 0.0
    if "weather_misery" not in f:
        f["weather_misery"] = 0.0

    return f


def predict_all(dt, h3_index=None, temperature=None, humidity=None,
                wind_speed=None, precipitation=None):
    feat_dict = build_features(dt, h3_index, temperature, humidity, wind_speed, precipitation)
    results = {}

    for target in MODELS:
        feat_list = FEATURES[target]
        X = np.array([[feat_dict.get(ft, 0.0) for ft in feat_list]], dtype=np.float64)
        raw = float(MODELS[target].predict(X)[0])
        task = METADATA[target].get("task_type", "regression")

        if task == "regression":
            val = round(max(raw, 0.0))
        else:
            val = int(round(raw))

        results[target] = {
            "value": val,
            "raw": raw,
            "name": METADATA[target].get("target_name", target),
            "task": task
        }
    return results


def predict_batch(requests_list):
    if not requests_list:
        return []

    master_features = [
        build_features(
            r["pickup_datetime"],
            r.get("h3_index"),
            r.get("temperature"),
            r.get("humidity"),
            r.get("wind_speed"),
            r.get("precipitation"),
        )
        for r in requests_list
    ]

    model_preds = {}
    for target in MODELS:
        feat_list = FEATURES[target]
        task = METADATA[target].get("task_type", "regression")
        name = METADATA[target].get("target_name", target)

        X = np.array(
            [[mf.get(ft, 0.0) for ft in feat_list] for mf in master_features],
            dtype=np.float64
        )

        raw_preds = MODELS[target].predict(X)

        preds = []
        for raw in raw_preds:
            raw = float(raw)
            if task == "regression":
                val = round(max(raw, 0.0))
            else:
                val = int(round(raw))
            preds.append({
                "value": val,
                "raw": raw,
                "name": name,
                "task": task
            })
        model_preds[target] = preds

    n = len(requests_list)
    return [{target: model_preds[target][i] for target in MODELS} for i in range(n)]


# ---------- FastAPI ----------
class PredReq(BaseModel):
    h3_index: Optional[str] = None
    pickup_datetime: dt_type = Field(...)
    temperature: Optional[float] = None
    humidity: Optional[float] = None
    wind_speed: Optional[float] = None
    precipitation: Optional[float] = None

class BatchReq(BaseModel):
    predictions: List[PredReq]

app = FastAPI(title="Chicago Ride Demand API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True,
                   allow_methods=["*"], allow_headers=["*"])

@app.get("/health")
async def health():
    return {
        "status": "healthy",
        "model_loaded": True,
        "models_loaded": list(MODELS.keys()),
        "model_count": len(MODELS),
    }

@app.post("/predict")
async def api_predict(req: PredReq):
    t0 = time.time()
    results = predict_all(
        req.pickup_datetime,
        req.h3_index,
        req.temperature,
        req.humidity,
        req.wind_speed,
        req.precipitation,
    )
    ms = (time.time() - t0) * 1000
    t3 = results.get("T3", {"value": 0, "raw": 0.0})

    return {
        "h3_index": req.h3_index,
        "pickup_datetime": req.pickup_datetime.isoformat(),
        "predictions": results,
        "predicted_rides": t3["value"],
        "predicted_rides_raw": t3["raw"],
        "model_info": {
            "inference_ms": round(ms, 2),
            "models": list(results.keys())
        }
    }

@app.post("/predict/batch")
async def api_batch(req: BatchReq):
    t0 = time.time()
    req_dicts = [
        {
            "pickup_datetime": r.pickup_datetime,
            "h3_index": r.h3_index,
            "temperature": r.temperature,
            "humidity": r.humidity,
            "wind_speed": r.wind_speed,
            "precipitation": r.precipitation,
        }
        for r in req.predictions
    ]

    batch_results = predict_batch(req_dicts)
    ms = (time.time() - t0) * 1000

    responses = []
    for i, results in enumerate(batch_results):
        r = req.predictions[i]
        t3 = results.get("T3", {"value": 0, "raw": 0.0})
        responses.append({
            "h3_index": r.h3_index,
            "pickup_datetime": r.pickup_datetime.isoformat(),
            "predictions": results,
            "predicted_rides": t3["value"],
            "predicted_rides_raw": t3["raw"],
            "model_info": {}
        })

    return {
        "predictions": responses,
        "count": len(responses),
        "model_info": {
            "total_ms": round(ms, 2),
            "avg_ms": round(ms / max(len(responses), 1), 2)
        }
    }


def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("Starting server...")
for i in range(10):
    time.sleep(1)
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=3)
        if r.status_code == 200:
            print("✅ API running at http://127.0.0.1:8000")
            break
    except:
        print(f"waiting... {i+1}")

✅ T1: 58 features
✅ T2: 55 features
✅ T3: 55 features
✅ T4: 58 features
✅ Feature cache loaded
   h3_hour: 13008
   h3: 542
   city_hour: 24
✅ Models warmed up
Starting server...
✅ API running at http://127.0.0.1:8000
